# F1 Lap Time Prediction - Holt Exponential Smoothing

I will compare simple exponential smoothing with the same-lap baseline from `04_linearegression_model.ipynb`.
This is the baseline model for the time-series forecasting models.

## Imports and loading data

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

laps = pd.read_pickle("../data/prepared_race_laps_2022_2025.pkl")

display(
    laps.groupby("Season").agg(
        Races=("RoundNumber", "nunique"),
        LapRecords=("LapNumber", "size"),
        EligiblePairs=("EligiblePair", "sum")
    )
)

print(f"Total eligible pairs: {laps['EligiblePair'].sum():,}")
laps_eligible = laps.loc[laps["EligiblePair"]].copy()

,Races,LapRecords,EligiblePairs
Season,,,
2022,22,23577,16971
2023,22,24420,18902
2024,24,26604,21394
2025,24,26689,20969


Total eligible pairs: 78,236


## Data Splitting

In [9]:
train_laps=laps_eligible[laps_eligible["Season"]<2024].copy()
validation_laps=laps_eligible[laps_eligible["Season"]==2024].copy()
test_laps=laps_eligible[laps_eligible["Season"]==2025].copy()


Data is splitted in this way, since in these years the cars are the closest to each other.

## Function for next-lap forecasts and Modelling

In [10]:
from statsmodels.tsa.holtwinters import Holt
keys=["Season", "RoundNumber","DriverNumber"]

def forecast_holt(data, alpha, beta):
    usable = (
        data["LapTimeSeconds"].notna()
        & data["IsPitLap"].eq(False)
        & data["TrackStatus"].eq("1")
        & data["IsAccurate"].eq(True)
    )

    results = (
        data.loc[usable]
        .sort_values(keys + ["LapNumber"])
        .copy()
    )

    results["Segment"] = (
        results.groupby(keys)["LapNumber"]
        .diff().ne(1).cumsum()
    )

    results["Prediction"] = float("nan")

    for _, sequence in results.groupby(keys + ["Segment"]):
        y = sequence["LapTimeSeconds"].to_numpy()

        if len(y) == 1:
            continue

        model = Holt(
            y,
            initialization_method="known",
            initial_level=y[0],
            initial_trend=0.0,
            damped_trend=False
        )

        fitted = model.fit(
            smoothing_level=alpha,
            smoothing_trend=beta,
            optimized=False
        )

        results.loc[sequence.index, "Prediction"] = (
            fitted.level + fitted.trend
        )

    return results.loc[results["EligiblePair"]].copy()

The cell written above gets the function to check if laps are following the respected rules of being integrated into race pace and then we get the results and create the model for our project.

## Training - selecting the smoothing parameter

In [11]:
train_history=laps.loc[laps["Season"]<2024].copy()
alpha_values=[0.1,0.3,0.5,0.7,0.9,1.0]
beta_values = [0.0, 0.01, 0.05, 0.1]
training_results=[]

for alpha in alpha_values:
    for beta in beta_values:
        train_predictions=forecast_holt(train_history, alpha=alpha, beta=beta)

        train_mae = mean_absolute_error(
        train_predictions["NextLapTimeSeconds"],
        train_predictions["Prediction"]
    )

        training_results.append({
        "Alpha": alpha,
        "Beta": beta,
        "TrainingMAE":train_mae
    })

training_results=pd.DataFrame(training_results)
display(training_results.sort_values("TrainingMAE").head(10))

,Alpha,Beta,TrainingMAE
20,1.0,0.00,0.489493
21,1.0,0.01,0.491468
16,0.9,0.00,0.492030
17,0.9,0.01,0.493355
22,1.0,0.05,0.509931
18,0.9,0.05,0.510161
13,0.7,0.01,0.514486
12,0.7,0.00,0.514727
19,0.9,0.10,0.525728
23,1.0,0.10,0.529470


The optimal Alpha and Beta on training give us the same-lap baseline. On validation I selected alpha=0.9 and beta=0, as these settings achieved the lowest validation MAE.

## Validation

In [12]:
validation_history=laps.loc[laps["Season"]==2024].copy()
alpha_values=[0.1,0.3,0.5,0.7,0.9,1.0]
beta_values = [0.0, 0.01, 0.05, 0.1]
validation_results=[]

for alpha in alpha_values:
    for beta in beta_values:
        validation_predictions=forecast_holt(validation_history, alpha=alpha, beta=beta)

        validation_mae = mean_absolute_error(
        validation_predictions["NextLapTimeSeconds"],
        validation_predictions["Prediction"]
    )

        validation_results.append({
        "Alpha": alpha,
        "Beta": beta,
        "ValidationMAE":validation_mae
    })

validation_results=pd.DataFrame(validation_results)
display(validation_results.sort_values("ValidationMAE").head(10))

,Alpha,Beta,ValidationMAE
16,0.9,0.00,0.460078
17,0.9,0.01,0.460396
13,0.7,0.01,0.463943
20,1.0,0.00,0.464507
12,0.7,0.00,0.464646
21,1.0,0.01,0.465289
18,0.9,0.05,0.466165
14,0.7,0.05,0.468011
22,1.0,0.05,0.472693
19,0.9,0.10,0.474294


Using the same combinations of alpha-beta on the validation split, alpha=0.9 and beta=0 achieved the lowest validation MAE. With the initial trend set to zero, estimated lap-times will get smoother, without extrapolating a trend.

In [13]:
best_parameters = validation_results.loc[
    validation_results["ValidationMAE"].idxmin()
]

selected_alpha = float(best_parameters["Alpha"])
selected_beta = float(best_parameters["Beta"])

errors = forecast_holt(
    validation_history,
    alpha=selected_alpha,
    beta=selected_beta
)

actual = errors["NextLapTimeSeconds"]
predictions = errors["Prediction"]
baseline_predictions = errors["LapTimeSeconds"]

comparison = pd.DataFrame({
    "Model": ["Selected smoothing model", "Same-lap baseline"],
    "MAE": [
        mean_absolute_error(actual, predictions),
        mean_absolute_error(actual, baseline_predictions)
    ],
    "RMSE": [
        mean_squared_error(actual, predictions) ** 0.5,
        mean_squared_error(actual, baseline_predictions) ** 0.5
    ],
    "R2": [
        r2_score(actual,predictions),
        r2_score(actual,baseline_predictions)
    ]
})

print(f"Selected alpha: {selected_alpha}, beta: {selected_beta}")
print(f"Evaluated pairs: {len(errors):,}")
display(comparison.round(6))

Selected alpha: 0.9, beta: 0.0
Evaluated pairs: 21,394


,Model,MAE,RMSE,R2
0,Selected smoothing model,0.460078,1.043222,0.989300
1,Same-lap baseline,0.464507,1.051193,0.989136


We got slightly lower errors than the baseline, so our model is performing slightly better on validation.

## Performance by race

In [14]:
errors["ModelAbsoluteError"] = (
    errors["NextLapTimeSeconds"] - errors["Prediction"]
).abs()

errors["BaselineAbsoluteError"] = (
    errors["NextLapTimeSeconds"] - errors["LapTimeSeconds"]
).abs()

race_comparison = (
    errors.groupby(["RoundNumber", "EventName"])
    .agg(
        Pairs=("Prediction", "size"),
        ModelMAE=("ModelAbsoluteError", "mean"),
        BaselineMAE=("BaselineAbsoluteError", "mean")
    )
    .reset_index()
)

race_comparison["ImprovementSeconds"] = (
    race_comparison["BaselineMAE"] - race_comparison["ModelMAE"]
)

display(
    race_comparison.sort_values(
        "ImprovementSeconds", ascending=False
    ).round(6)
)

improved = (race_comparison["ImprovementSeconds"] > 0).sum()
print(f"Smoothing improved MAE in {improved} of {len(race_comparison)} races.")

,RoundNumber,EventName,Pairs,ModelMAE,BaselineMAE,ImprovementSeconds
7,8,Monaco Grand Prix,1134,0.664922,0.678518,0.013595
6,7,Emilia Romagna Grand Prix,1118,0.407719,0.420687,0.012968
9,10,Spanish Grand Prix,1144,0.437089,0.449397,0.012308
2,3,Australian Grand Prix,785,0.351896,0.364000,0.012104
14,15,Dutch Grand Prix,1308,0.379302,0.391346,0.012044
19,20,Mexico City Grand Prix,1021,0.385694,0.397385,0.011691
15,16,Italian Grand Prix,877,0.385490,0.395645,0.010155
0,1,Bahrain Grand Prix,931,0.303593,0.312539,0.008946
10,11,Austrian Grand Prix,1152,0.319109,0.327984,0.008875
23,24,Abu Dhabi Grand Prix,821,0.294152,0.302300,0.008147


Smoothing improved MAE in 19 of 24 races.


Even though the improvements were quite small, it's still great that this model has a decent performance (basically the same as the baseline). The high R2 is similar to the baseline, and the mean absolute error is around 0.46 seconds. We still need testing to assess performance on another year. This does not mean this model is bad; its suitability for real life depends on the precision needed.

## Explaining individual predictions
I will look into ten predictions from an uninterrupted lap sequence with beta(trend) set to zero. I will calculate this manually and check if it matches the model output

In [16]:
segment = errors["Segment"].value_counts().idxmax()
example = (
    errors.loc[errors["Segment"] == segment]
    .sort_values("LapNumber")
    .head(10)
    .copy()
)

level = example["LapTimeSeconds"].iloc[0]
previous_levels = []
manual_predictions = []

for lap_time in example["LapTimeSeconds"]:
    previous_levels.append(level)

    level = selected_alpha * lap_time + (1 - selected_alpha) * level
    manual_predictions.append(level)

example["PreviousLevel"] = previous_levels
example["ManualPrediction"] = manual_predictions
example["Difference"] = example["ManualPrediction"] - example["Prediction"]

display(example[[
    "LapNumber",
    "LapTimeSeconds",
    "PreviousLevel",
    "ManualPrediction",
    "Prediction",
    "Difference"
]].round(6))

,LapNumber,LapTimeSeconds,PreviousLevel,ManualPrediction,Prediction,Difference
64491,16.0,75.932,75.932000,75.932000,75.932000,0.0
64492,17.0,76.203,75.932000,76.175900,76.175900,0.0
64493,18.0,76.066,76.175900,76.076990,76.076990,0.0
64494,19.0,75.867,76.076990,75.887999,75.887999,0.0
64495,20.0,75.953,75.887999,75.946500,75.946500,0.0
64496,21.0,76.281,75.946500,76.247550,76.247550,0.0
64497,22.0,77.053,76.247550,76.972455,76.972455,0.0
64498,23.0,76.658,76.972455,76.689445,76.689445,0.0
64499,24.0,76.652,76.689445,76.655745,76.655745,0.0
64500,25.0,76.387,76.655745,76.413874,76.413874,0.0


The manual calculations matched the model’s predictions for all ten laps. Each forecast uses 90% of the current lap time and 10% of the previous level. With beta=0 and an initial trend of zero, the model follows changes in pace without projecting a trend. This makes the calculation understandable, but does not mean the predictions are perfectly accurate.

## Testing Split

In [18]:
test_history = laps.loc[laps["Season"] == 2025].copy()

test_errors = forecast_holt(
    test_history,
    alpha=selected_alpha,
    beta=selected_beta
)

test_actual = test_errors["NextLapTimeSeconds"]
test_predictions = test_errors["Prediction"]
test_baseline_predictions = test_errors["LapTimeSeconds"]

test_comparison = pd.DataFrame({
    "Model": ["Selected smoothing model", "Same-lap baseline"],
    "MAE": [
        mean_absolute_error(test_actual, test_predictions),
        mean_absolute_error(test_actual, test_baseline_predictions)
    ],
    "RMSE": [
        mean_squared_error(test_actual, test_predictions) ** 0.5,
        mean_squared_error(test_actual, test_baseline_predictions) ** 0.5
    ],
    "R2": [
        r2_score(test_actual, test_predictions),
        r2_score(test_actual, test_baseline_predictions)
    ]
})

print(f"Selected alpha: {selected_alpha}, beta: {selected_beta}")
print(f"Evaluated test pairs: {len(test_errors):,}")
display(test_comparison.round(4))

Selected alpha: 0.9, beta: 0.0
Evaluated test pairs: 20,969


,Model,MAE,RMSE,R2
0,Selected smoothing model,0.3807,0.6677,0.9959
1,Same-lap baseline,0.3855,0.6812,0.9957


## Final Conclusion for this model
The smoothing model achieved a test MAE of **0.3807 seconds**, improving on the same-lap baseline by approximately **1.25%**. Manual calculations matched its predictions, howing that the model is quite well done and does not have any problems in overfitting/underfitting or data leakage even. Performance did not got worse from training to validation and testing, so there is no clear indication of overfitting. The improvement is small, but provides a useful starting point.

For context, a [similar F1 prediction project](https://github.com/jacopo887/ADA_project_JS#results) reports a test MAE of **0.4199 seconds using Ridge regression** for next-lap prediction within races. 

Of course different models, different splits, settings, parameters and inputs affect model's performance, however this was already a good improvement on other people's model. 
The best improvemements for next iteration would be create a ARIMA model to compare to this one and lookk more into detail on what is happening with the model using SHAP, MLFLOW or Optuna. It is really essential to check this out, before finalizing the project, selecting an end model for the app and conecting everything.